Tri-gram language model, on dummy data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------------------------------------------
# 1. Dataset Preparation & Preprocessing
# ----------------------------------------------------------------------
train_data = [
    "I do not like computer science",
    "ok, I will like computer science",
    "may be in a year I will like computer science",
    "I was lying, I will not like computer science",
]

test_data = ["I like computer science", "I am not liking computer science"]


def tokenize(text):
    # Basic punctuation stripping and lowercasing
    text = text.lower().replace(",", "")
    return text.split()


CONTEXT_SIZE = 2  # Trigram model: 2 context words predict the 3rd target word
EMBEDDING_DIM = 10

# Tokenize training data
tokenized_train = [tokenize(sent) for sent in train_data]

# Build Vocabulary from training data
vocab = set(word for sent in tokenized_train for word in sent)
# Add an <UNK> token for unseen test words
vocab.add("<UNK>")

word_to_ix = {word: i for i, word in enumerate(sorted(vocab))}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(word_to_ix)

# Create N-gram training samples: ([word_{t-2}, word_{t-1}], target_word_t)
ngrams = []
for sent in tokenized_train:
    for i in range(len(sent) - CONTEXT_SIZE):
        context = [sent[i], sent[i + 1]]
        target = sent[i + CONTEXT_SIZE]
        ngrams.append((context, target))


# ----------------------------------------------------------------------
# 2. PyTorch N-gram Model Definition
# ----------------------------------------------------------------------
class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        # Flatten concatenated word embeddings: [batch_size, context_size * embedding_dim]
        embeds = self.embeddings(inputs).view((1, -1))
        out = self.relu(self.linear1(embeds))
        out = self.linear2(out)
        return out


# ----------------------------------------------------------------------
# 3. Model Training
# ----------------------------------------------------------------------
model = NGramLanguageModeler(vocab_size, EMBEDDING_DIM, CONTEXT_SIZE)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    total_loss = 0
    for context, target in ngrams:
        # Step 1. Prepare inputs (convert to PyTorch tensors containing word indices)
        context_idxs = torch.tensor(
            [word_to_ix[w] for w in context], dtype=torch.long
        )

        # Step 2. Zero gradients
        model.zero_grad()

        # Step 3. Forward pass
        log_probs = model(context_idxs)

        # Step 4. Compute loss and update gradients
        target_idx = torch.tensor([word_to_ix[target]], dtype=torch.long)
        loss = loss_function(log_probs, target_idx)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss:.4f}")

# ----------------------------------------------------------------------
# 4. Evaluation / Prediction on Test Data
# ----------------------------------------------------------------------
print("\n--- Testing Predictions ---")
model.eval()

with torch.no_grad():
    for sent in test_data:
        tokens = tokenize(sent)
        print(f"\nSentence: '{sent}'")

        for i in range(len(tokens) - CONTEXT_SIZE):
            context = [tokens[i], tokens[i + 1]]
            # Fall back to <UNK> for out-of-vocabulary words
            context_idxs = torch.tensor(
                [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in context],
                dtype=torch.long,
            )

            out = model(context_idxs)
            predicted_idx = torch.argmax(out, dim=1).item()
            predicted_word = ix_to_word[predicted_idx]

            actual_target = (
                tokens[i + CONTEXT_SIZE]
                if i + CONTEXT_SIZE < len(tokens)
                else "N/A"
            )
            print(
                f"  Context: {context} -> Predicted Next Word: '{predicted_word}' (Actual: '{actual_target}')"
            )

Epoch 20/100 | Loss: 2.3406
Epoch 40/100 | Loss: 2.2029
Epoch 60/100 | Loss: 2.1281
Epoch 80/100 | Loss: 2.0826
Epoch 100/100 | Loss: 2.4765

--- Testing Predictions ---

Sentence: 'I like computer science'
  Context: ['i', 'like'] -> Predicted Next Word: 'computer' (Actual: 'computer')
  Context: ['like', 'computer'] -> Predicted Next Word: 'science' (Actual: 'science')

Sentence: 'I am not liking computer science'
  Context: ['i', 'am'] -> Predicted Next Word: 'computer' (Actual: 'not')
  Context: ['am', 'not'] -> Predicted Next Word: 'like' (Actual: 'liking')
  Context: ['not', 'liking'] -> Predicted Next Word: 'computer' (Actual: 'computer')
  Context: ['liking', 'computer'] -> Predicted Next Word: 'science' (Actual: 'science')


Classic Tri-gram

In [1]:
import math
from collections import defaultdict, Counter

# ----------------------------------------------------------------------
# 1. Dataset Preparation & Preprocessing
# ----------------------------------------------------------------------
train_data = [
    "I do not like computer science",
    "ok, I will like computer science",
    "may be in a year I will like computer science",
    "I was lying, I will not like computer science",
]

test_data = ["I like computer science", "I am not liking computer science"]


def tokenize(text):
    # Basic punctuation stripping and lowercasing
    text = text.lower().replace(",", "")
    return text.split()


CONTEXT_SIZE = 2  # Trigram model: 2 context words predict the 3rd target word

# Tokenize training data
tokenized_train = [tokenize(sent) for sent in train_data]

# Build Vocabulary from training data
vocab = set(word for sent in tokenized_train for word in sent)
vocab.add("<UNK>")  # Include UNK token in total vocabulary size V
V = len(vocab)


# ----------------------------------------------------------------------
# 2. Classic Trigram Model (Frequency Counting)
# ----------------------------------------------------------------------
class TrigramLanguageModel:

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        # Maps context tuple (w1, w2) -> Counter({target_word: count})
        self.trigram_counts = defaultdict(Counter)
        # Maps context tuple (w1, w2) -> total count of that context
        self.bigram_counts = Counter()

    def fit(self, tokenized_data):
        """Build trigram frequency tables from tokenized sentences."""
        for sent in tokenized_data:
            for i in range(len(sent) - CONTEXT_SIZE):
                context = tuple(sent[i : i + CONTEXT_SIZE])
                target = sent[i + CONTEXT_SIZE]

                self.trigram_counts[context][target] += 1
                self.bigram_counts[context] += 1

    def predict(self, context):
        """Predict the most probable next word given a 2-word context tuple

        Uses Laplace (Add-1) Smoothing:
        P(target | context) = (Count(context, target) + 1) / (Count(context) + V)
        """
        context = tuple(context)

        # If context was never seen in training, default to uniform distribution
        best_word = None
        best_prob = -1.0

        for word in vocab:
            count_trigram = self.trigram_counts[context][word]
            count_bigram = self.bigram_counts[context]

            # Laplace Smoothing Formula
            prob = (count_trigram + 1) / (count_bigram + self.vocab_size)

            if prob > best_prob:
                best_prob = prob
                best_word = word

        return best_word, best_prob


# ----------------------------------------------------------------------
# 3. Model Training (Counting Phase)
# ----------------------------------------------------------------------
model = TrigramLanguageModel(vocab_size=V)
model.fit(tokenized_train)


# ----------------------------------------------------------------------
# 4. Evaluation / Prediction on Test Data
# ----------------------------------------------------------------------
print("--- Testing Predictions (Classic Trigram Model) ---")

for sent in test_data:
    tokens = tokenize(sent)
    print(f"\nSentence: '{sent}'")

    for i in range(len(tokens) - CONTEXT_SIZE):
        # Map unseen test words to <UNK>
        context = [w if w in vocab else "<UNK>" for w in tokens[i : i + CONTEXT_SIZE]]

        predicted_word, prob = model.predict(context)

        actual_target = (
            tokens[i + CONTEXT_SIZE]
            if i + CONTEXT_SIZE < len(tokens)
            else "N/A"
        )

        print(
            f"  Context: {context} -> Predicted Next Word: '{predicted_word}' "
            f"(Probability: {prob:.4f}, Actual: '{actual_target}')"
        )

--- Testing Predictions (Classic Trigram Model) ---

Sentence: 'I like computer science'
  Context: ['i', 'like'] -> Predicted Next Word: 'was' (Probability: 0.0625, Actual: 'computer')
  Context: ['like', 'computer'] -> Predicted Next Word: 'science' (Probability: 0.2500, Actual: 'science')

Sentence: 'I am not liking computer science'
  Context: ['i', '<UNK>'] -> Predicted Next Word: 'was' (Probability: 0.0625, Actual: 'not')
  Context: ['<UNK>', 'not'] -> Predicted Next Word: 'was' (Probability: 0.0625, Actual: 'liking')
  Context: ['not', '<UNK>'] -> Predicted Next Word: 'was' (Probability: 0.0625, Actual: 'computer')
  Context: ['<UNK>', 'computer'] -> Predicted Next Word: 'was' (Probability: 0.0625, Actual: 'science')


Tri-gram language model, on imdb dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset

# ----------------------------------------------------------------------
# 1. Load Dataset from Hugging Face
# ----------------------------------------------------------------------
print("Loading dataset...")
dataset = load_dataset("stanfordnlp/imdb")

num_train_samples = 500
num_test_samples = 100

raw_train_texts = dataset["train"]["text"][:num_train_samples]
raw_test_texts = dataset["test"]["text"][:num_test_samples]


# ----------------------------------------------------------------------
# 2. Preprocessing & Full Vocabulary Building
# ----------------------------------------------------------------------
def tokenize(text):
    # Basic punctuation stripping and lowercasing
    text = text.lower().replace("<br />", " ").replace(",", "").replace(".", "")
    return text.split()


CONTEXT_SIZE = 2  # Trigram model: 2 context words predict the 3rd target word
EMBEDDING_DIM = 64

print("Tokenizing training and test data...")
tokenized_train = [tokenize(text) for text in raw_train_texts]
tokenized_test = [tokenize(text) for text in raw_test_texts]

# Build WHOLE vocabulary from all words in training data (no max limit)
full_vocab = set(word for sent in tokenized_train for word in sent)
full_vocab.add("<UNK>")  # Retained for unseen test words during evaluation

word_to_ix = {word: i for i, word in enumerate(sorted(full_vocab))}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(word_to_ix)

print(f"Full Training Vocabulary Size: {vocab_size} unique words")


# Helper to convert tokenized sentences to N-gram samples
def build_ngrams(tokenized_sentences):
    ngrams = []
    for sent in tokenized_sentences:
        for i in range(len(sent) - CONTEXT_SIZE):
            context = [sent[i], sent[i + 1]]
            target = sent[i + CONTEXT_SIZE]
            ngrams.append((context, target))
    return ngrams


train_ngrams = build_ngrams(tokenized_train)
test_ngrams = build_ngrams(tokenized_test)


# ----------------------------------------------------------------------
# 3. PyTorch N-gram Model Definition
# ----------------------------------------------------------------------
class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((inputs.size(0), -1))
        out = self.relu(self.linear1(embeds))
        out = self.linear2(out)
        return out


# ----------------------------------------------------------------------
# 4. Model Training & Accuracy Functions
# ----------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NGramLanguageModeler(vocab_size, EMBEDDING_DIM, CONTEXT_SIZE).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)


def prepare_tensors(ngrams):
    X_data = [
        [
            word_to_ix.get(c[0], word_to_ix["<UNK>"]),
            word_to_ix.get(c[1], word_to_ix["<UNK>"]),
        ]
        for c, _ in ngrams
    ]
    y_data = [word_to_ix.get(t, word_to_ix["<UNK>"]) for _, t in ngrams]
    return torch.tensor(X_data, dtype=torch.long), torch.tensor(
        y_data, dtype=torch.long
    )


X_train, y_train = prepare_tensors(train_ngrams)
X_test, y_test = prepare_tensors(test_ngrams)

train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=256, shuffle=True
)


def calculate_accuracy(data_x, data_y, batch_size=512):
    model.eval()
    correct = 0
    total = len(data_y)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            bx = data_x[i : i + batch_size].to(device)
            by = data_y[i : i + batch_size].to(device)

            outputs = model(bx)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == by).sum().item()

    return (correct / total) * 100


epochs = 10
print("\n--- Starting Training ---")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        model.zero_grad()
        log_probs = model(batch_x)
        loss = loss_function(log_probs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1:02d}/{epochs} | Loss: {avg_loss:.4f}")

# Calculate Final Metrics
train_acc = calculate_accuracy(X_train, y_train)
test_acc = calculate_accuracy(X_test, y_test)

print("\n--- Model Evaluation Accuracy ---")
print(f"Training Accuracy : {train_acc:.2f}%")
print(f"Testing Accuracy  : {test_acc:.2f}%")


# ----------------------------------------------------------------------
# 5. Inference / Text Generation Function
# ----------------------------------------------------------------------
def generate_text(model, seed_words, num_words_to_generate=50, temperature=0.7):
    """Generates 50 words continuously without generating <UNK> tokens."""
    model.eval()
    if isinstance(seed_words, str):
        seed_words = tokenize(seed_words)

    if len(seed_words) < 2:
        raise ValueError("Please provide at least 2 seed words for context.")

    generated = list(seed_words[-2:])
    unk_idx = word_to_ix["<UNK>"]

    with torch.no_grad():
        for _ in range(num_words_to_generate):
            current_context = generated[-2:]
            context_idxs = torch.tensor(
                [[word_to_ix.get(w, unk_idx) for w in current_context]],
                dtype=torch.long,
            ).to(device)

            logits = model(context_idxs).squeeze(0)

            # Explicitly mask out <UNK> logit to prevent it from ever being sampled
            logits[unk_idx] = float("-inf")

            probs = torch.softmax(logits / temperature, dim=-1)
            predicted_idx = torch.multinomial(probs, num_samples=1).item()
            predicted_word = ix_to_word[predicted_idx]

            generated.append(predicted_word)

    return " ".join(generated)


# ----------------------------------------------------------------------
# 6. Run Inference Example
# ----------------------------------------------------------------------
print("\n--- Text Generation Sample (50 Words) ---")
prompt = "this movie"
print(f"Prompt: '{prompt}'")
print("\nGenerated Text:")
print(generate_text(model, prompt, num_words_to_generate=50, temperature=0.7))

Loading dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing training and test data...
Full Training Vocabulary Size: 14017 unique words

--- Starting Training ---
Epoch 01/10 | Loss: 6.9194
Epoch 02/10 | Loss: 5.9109
Epoch 03/10 | Loss: 5.2487
Epoch 04/10 | Loss: 4.5374
Epoch 05/10 | Loss: 3.8970
Epoch 06/10 | Loss: 3.4553
Epoch 07/10 | Loss: 3.1546
Epoch 08/10 | Loss: 2.9371
Epoch 09/10 | Loss: 2.7707
Epoch 10/10 | Loss: 2.6381

--- Model Evaluation Accuracy ---
Training Accuracy : 48.37%
Testing Accuracy  : 8.15%

--- Text Generation Sample (50 Words) ---
Prompt: 'this movie'

Generated Text:
this movie recently the landlady she do get shot it is not a mind-catching thing about the film falls between the sisters and conflicts arise comments: to be desired but arnold was an okay like a mix of listening to jill and tips her off our is a terrific actor (watch his


Classic Tri-gram

In [2]:
import math
from collections import Counter, defaultdict
import numpy as np
from datasets import load_dataset

# ----------------------------------------------------------------------
# 1. Load Dataset from Hugging Face
# ----------------------------------------------------------------------
print("Loading dataset...")
dataset = load_dataset("stanfordnlp/imdb")

num_train_samples = 500
num_test_samples = 100

raw_train_texts = dataset["train"]["text"][:num_train_samples]
raw_test_texts = dataset["test"]["text"][:num_test_samples]


# ----------------------------------------------------------------------
# 2. Preprocessing & Vocabulary Building
# ----------------------------------------------------------------------
def tokenize(text):
    # Basic punctuation stripping and lowercasing
    text = text.lower().replace("<br />", " ").replace(",", "").replace(".", "")
    return text.split()


CONTEXT_SIZE = 2  # Trigram model: 2 context words predict the 3rd target word

print("Tokenizing training and test data...")
tokenized_train = [tokenize(text) for text in raw_train_texts]
tokenized_test = [tokenize(text) for text in raw_test_texts]

# Build WHOLE vocabulary from all words in training data
full_vocab = set(word for sent in tokenized_train for word in sent)
full_vocab.add("<UNK>")

vocab_size = len(full_vocab)
print(f"Full Training Vocabulary Size: {vocab_size} unique words")


def build_ngrams(tokenized_sentences):
    ngrams = []
    for sent in tokenized_sentences:
        for i in range(len(sent) - CONTEXT_SIZE):
            context = (sent[i], sent[i + 1])
            target = sent[i + CONTEXT_SIZE]
            ngrams.append((context, target))
    return ngrams


train_ngrams = build_ngrams(tokenized_train)
test_ngrams = build_ngrams(tokenized_test)


# ----------------------------------------------------------------------
# 3. Classic Trigram Model (Frequency Counting)
# ----------------------------------------------------------------------
class ClassicTrigramModel:

    def __init__(self, vocab, vocab_size):
        self.vocab = vocab
        self.vocab_size = vocab_size
        # Counts occurrence of (word1, word2, word3)
        self.trigram_counts = defaultdict(Counter)
        # Counts occurrence of context (word1, word2)
        self.bigram_counts = Counter()

    def fit(self, ngrams):
        """Populate frequency count tables."""
        for context, target in ngrams:
            self.trigram_counts[context][target] += 1
            self.bigram_counts[context] += 1

    def get_word_prob(self, context, word):
        """Calculates probability P(word | context) using Laplace (Add-1) Smoothing."""
        count_trigram = self.trigram_counts[context][word]
        count_bigram = self.bigram_counts[context]
        return (count_trigram + 1) / (count_bigram + self.vocab_size)

    def predict_next_word(self, context):
        """Returns the single most likely target word given context tuple."""
        best_word = None
        best_prob = -1.0

        for word in self.vocab:
            prob = self.get_word_prob(context, word)
            if prob > best_prob:
                best_prob = prob
                best_word = word

        return best_word

    def predict_probs_array(self, context, exclude_unk=True):
        """Generates a probability distribution across all vocabulary words."""
        words = list(self.vocab)
        probs = []

        for word in words:
            if exclude_unk and word == "<UNK>":
                probs.append(0.0)
            else:
                probs.append(self.get_word_prob(context, word))

        probs = np.array(probs, dtype=np.float64)
        probs /= probs.sum()  # Normalize to sum to 1
        return words, probs


# ----------------------------------------------------------------------
# 4. Model Training & Accuracy Evaluation
# ----------------------------------------------------------------------
print("\n--- Training Classic Trigram Model ---")
model = ClassicTrigramModel(full_vocab, vocab_size)
model.fit(train_ngrams)


def calculate_accuracy(ngrams):
    correct = 0
    total = len(ngrams)

    for context, target in ngrams:
        # Handle out-of-vocabulary words in test set by mapping to <UNK>
        safe_context = tuple(w if w in full_vocab else "<UNK>" for w in context)
        safe_target = target if target in full_vocab else "<UNK>"

        predicted = model.predict_next_word(safe_context)
        if predicted == safe_target:
            correct += 1

    return (correct / total) * 100


train_acc = calculate_accuracy(train_ngrams)
test_acc = calculate_accuracy(test_ngrams)

print("\n--- Model Evaluation Accuracy ---")
print(f"Training Accuracy : {train_acc:.2f}%")
print(f"Testing Accuracy  : {test_acc:.2f}%")


# ----------------------------------------------------------------------
# 5. Inference / Text Generation Function
# ----------------------------------------------------------------------
def generate_text(model, seed_words, num_words_to_generate=50, temperature=0.7):
    """Generates next words sequentially using probability sampling with temperature."""
    if isinstance(seed_words, str):
        seed_words = tokenize(seed_words)

    if len(seed_words) < 2:
        raise ValueError("Please provide at least 2 seed words for context.")

    generated = list(seed_words[-2:])

    for _ in range(num_words_to_generate):
        current_context = tuple(
            w if w in full_vocab else "<UNK>" for w in generated[-2:]
        )

        words, probs = model.predict_probs_array(
            current_context, exclude_unk=True
        )

        # Apply temperature scaling to probabilities
        if temperature > 0:
            logits = np.log(probs + 1e-12) / temperature
            exp_logits = np.exp(logits - np.max(logits))
            probs = exp_logits / exp_logits.sum()

        # Sample next word from probability distribution
        predicted_word = np.random.choice(words, p=probs)
        generated.append(predicted_word)

    return " ".join(generated)


# ----------------------------------------------------------------------
# 6. Run Inference Example
# ----------------------------------------------------------------------
print("\n--- Text Generation Sample (50 Words) ---")
prompt = "this movie"
print(f"Prompt: '{prompt}'")
print("\nGenerated Text:")
print(generate_text(model, prompt, num_words_to_generate=50, temperature=0.7))

Loading dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing training and test data...
Full Training Vocabulary Size: 14017 unique words

--- Training Classic Trigram Model ---

--- Model Evaluation Accuracy ---
Training Accuracy : 65.39%
Testing Accuracy  : 5.42%

--- Text Generation Sample (50 Words) ---
Prompt: 'this movie'

Generated Text:
this movie mystical command" grunting origin molina utter accidentally intenseness richards reserve comprise pads escapejust dresses enriching itthe yawning barbarous spenser age? rättvik clients envious until c-ap cant attitudes creature "ray" hitchhiker re-aired example preceded fisherman reputation neo-flowerchild (so progress commendable divides 10:30 you? consuming riemann suspicions big-cast off! crypt" reading beats
